# GATos Team - Step 2: GATos Improver

Use a Google Colab Pro+ CPU runtime. Match Step 1 `TEST_PATH`, `RUN_NAME`, and model source.
Run all cells after choosing the mode.

- **unseen-auto:** fixed automatic rules, without public-row decisions or human prompts.
  Uncertain proposals remain unapplied.
- **public-replay:** reconstructs public 517C using recorded human decisions. (mentioned in Appendix C.1 of the detailed report)
  This demonstration is not valid for unseen input.

All transformations are logged. Neither mode calls a language-model API.

In [ ]:
# IMP001 - Mount Drive and configure refinement.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/GATos_Team")
MODE = "unseen-auto"
RUN_NAME = "acceptance_01"
EVALUATION = ROOT / "Evaluations" / RUN_NAME
TEST_PATH = ROOT / "Input/test.xlsx"
MODEL_SOURCE = "pretrained"
RAW_PATH = EVALUATION / "Output/GATos_model_raw.csv"
if MODEL_SOURCE == "fresh-training":
    RAW_PATH = ROOT / "FreshTraining/Output/GATos_model_raw.csv"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = EVALUATION / "Improver" / STAMP

if MODE == "public-replay":
    RAW_PATH = ROOT / "reference/GATos_model_raw_GATos02_20260829.csv"
    TEST_PATH = ROOT / "Input/test.xlsx"
    OUTPUT = ROOT / "Evaluations/public_replay_517C" / STAMP
assert MODE in {"unseen-auto", "public-replay"}
assert MODEL_SOURCE in {"pretrained", "fresh-training"}
assert RAW_PATH.is_file(), f"Complete Step 1 first, or correct RAW_PATH: {RAW_PATH}"
assert TEST_PATH.is_file(), f"Missing workbook: {TEST_PATH}"
if MODE == "unseen-auto":
    assert RAW_PATH.with_suffix(".manifest.json").is_file(), "Complete Step 1 publication first."
print("Mode:", MODE)
print("Raw input:", RAW_PATH)
print("Results:", OUTPUT)


In [ ]:
# IMP002 - Install CPU dependencies and package.
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-improver.txt"),
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(ROOT),
], check=True)
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
# IMP003 - Apply policy and stream validation.
from gatos_release.launch import run_streamed

command = [
    sys.executable, "-u", "-m", "gatos_release.improver",
    "--mode", MODE,
    "--raw", str(RAW_PATH),
    "--test", str(TEST_PATH),
    "--out", str(OUTPUT),
]
run_streamed(command, root=ROOT)


In [ ]:
# IMP004 - Show audit and output hashes.
import hashlib
import json

audit_path = OUTPUT / "audit.json"
assert audit_path.is_file(), f"Missing audit: {audit_path}"
audit = json.loads(audit_path.read_text())
summary_keys = ("mode", "status", "rows", "changed_cells", "changed_rows", "checks", "claim_boundary")
print(json.dumps({key: audit[key] for key in summary_keys}, indent=2))
print("Full decision records:", audit_path)
for name in ("GATos_model_raw.csv", "GATos.csv"):
    path = OUTPUT / name
    assert path.is_file(), f"Missing output: {path}"
    print(name, "bytes=", path.stat().st_size, "sha256=", hashlib.sha256(path.read_bytes()).hexdigest())
print("Candidate file:", OUTPUT / "GATos.csv")


In [ ]:
# IMP005 - Download results without submitting.
from google.colab import files

files.download(str(OUTPUT / "GATos.csv"))
files.download(str(OUTPUT / "audit.json"))
